#### **01-backtest**

A rule written as a class, run over a year of hourly bars on a simulated venue
that charges for every trade, then decomposed and charted.

The rule is a moving average crossover, which is the simplest thing that counts
as a strategy. **The point is not the rule.** It is what a run costs, how you
find out where the money went, and how to drive the chart that shows it.

#### **Setup**

The environment is built before this cell runs. `docker compose up` installs
both libraries and starts a router. A pip install next to a router you started
yourself does the same thing.

So this cell checks and opens the client. It installs nothing and starts
nothing. If either library is missing, or nothing answers on the router
address, it stops here and prints the command to run.

Two names are needed this time. The [client](https://github.com/atOCEANO/exchange-router-service/blob/main/.Documentation/Python_SDK.md)
fetches the bars and [emsl](https://github.com/atOCEANO/embeddable-market-simulation-library)
simulates against them.

In [1]:
import os
import urllib.request

ROUTER_URL = os.environ.get(key="ROUTER_URL", default="http://127.0.0.1:8040")

try:
    import emsl
    import exchange_router_client
except ImportError as error:
    raise ImportError(f"{error.name} is missing; run 'docker compose up' from "
                      f"this repository, or pip install -r requirements.txt") from None


def router_is_up(url):
    try:
        with urllib.request.urlopen(url=f"{url}/status", timeout=2.0) as response:
            return response.status == 200
    except Exception:
        return False


if not router_is_up(url=ROUTER_URL):
    raise RuntimeError(f"no router answering on {ROUTER_URL}; run 'docker compose "
                       f"up' from this repository, or start one yourself and point "
                       f"ROUTER_URL at it")

client = exchange_router_client.ExchangeRouterClient(base_url=ROUTER_URL)

print("service    ", client.get_version(), "at", ROUTER_URL)
print("client     ", exchange_router_client.__version__)
print("emsl       ", emsl.__version__)

service     2.5.6 at http://exchange-router-service:8040
client      5.1.0
emsl        1.2.0


#### **The candles**

One year of hourly bars, pinned so the run is over the same bars every time.
**`start` is the end of the window and the router walks backwards from it**, so
this asks for the year that ends on 1 January 2026, which is 2025.

**Hand emsl the frame, not a bare array.** It lines its arrays up by position
and never by the index, but it does read the index to work out how many bars a
year holds. Give it a plain numpy array instead and it warns, then annualizes
Sharpe, Sortino, volatility, CAGR and Calmar as though each bar were a day,
which on hourly data is wrong by a large factor.

A frame carries metadata, not just columns. What is constant across the
whole request lives in `candles.attrs`: the venue, the symbol, the quote
asset, the unit volume is counted in, and the schema number the router and
the client agree on. It is printed here because **emsl does not read
attrs**. The volume unit in particular is something you have to know rather
than something that travels with the numbers into the simulation.

In [2]:
import datetime

EXCHANGE = "binance"
MARKET   = "spot"
SYMBOL   = "BTCUSDT"
INTERVAL = "1h"

ANCHOR = int(datetime.datetime(year=2026, month=1, day=1,
                               tzinfo=datetime.timezone.utc).timestamp() * 1000)
BARS   = 8760

candles = client.get_candles(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbol=SYMBOL,
    interval=INTERVAL,
    limit=BARS,
    start=ANCHOR,
)

print(f"{len(candles)} bars, {candles.index[0]} to {candles.index[-1]}")
print(f"columns  {', '.join(candles.columns)}")

for key in ("exchange", "market_type", "symbol", "quote", "volume_unit",
            "schema_version"):
    print(f"  {key:<16} {candles.attrs[key]}")

8760 bars, 2025-01-01 01:00:00+00:00 to 2026-01-01 00:00:00+00:00
columns  open, high, low, close, volume, volume_usd
  exchange         binance
  market_type      spot
  symbol           BTCUSDT
  quote            USDT
  volume_unit      base
  schema_version   3


#### **The rule**

A `Strategy` is Python's own constructor plus two methods the engine calls,
and the three run at different times. `__init__` runs when you build the
object, before any data exists, so it holds the tunables and nothing else.
**`init` is emsl's hook**, run once at the start of a run, which is the first
moment the series exists and so the first moment an indicator can be
computed. **`next` runs once per bar** and places the orders.

The two names are four characters apart and easy to blur. The split is what
lets anything build many strategies from one class: a parameter search calls
the constructor with the values it sampled, then hands the object to the
engine, which calls `init` on it. Tunables in `__init__`, arrays in `init`.

Where the indicators get computed is a choice, and this rule makes the common
one.

Precomputing whole arrays in `init` is the fast path. Recomputing with
numpy on every bar makes your own per-bar code the bottleneck rather than the
engine. It is safe here only because every `emsl.ta` function is causal: entry
`i` uses bars up to `i` and no further. Write one that is not, a z-score over
the whole series, a min-max normalisation, any two-pass filter, and entry `i`
now carries information from bars after it. Reading `self.thing[i]` inside
`next` then looks at the future, and looks innocent doing it.

Computing inside `next` buys a guarantee instead. Slice
`engine.closes[:i + 1]` and the future is not in the array you are holding, so
there is no discipline left to forget. The slice is what does that, not the
method: `engine.closes` is a view of the whole series in `next` too, and
`engine.closes[i + 50]` is perfectly readable there.

Some indicators cannot go in `init` at all. Anything that reads the run
rather than the series: a stop anchored to the entry price, bars since entry, a
cooldown after a loss, a size that scales with current equity, an accumulation
that resets when the position goes flat. Those are functions of what the
strategy did, and that does not exist until it does it.

**`warmup` is the line people leave out.** It is the longest history any
decision reads, and `next` is not called until that many bars exist. Without
it, `closes[i - 60]` on bar 3 is a negative index, numpy resolves it from the
**end** of the array, and the rule reads prices from the future. The result
looks good and nothing raises. Here it is set in `init`, which emsl allows
because it reads it afterwards.

`crossover` is a cross and not a comparison. It is true only on the bar where
fast goes from below slow to above it, not on every bar it stays above. It
pads the warm-up with `False` rather than NaN, because `bool(nan)` is `True`,
so a NaN pad would fire the rule on its first bar.

`spread` is kept for the chart later. It is the rule's actual decision variable:
the crossings are that line touching zero.

`marks()` is not part of the Strategy interface. emsl never calls it and it
runs nothing during a backtest. It is here because a label and the parameter it
names belong together: write `"EMA 20"` into a chart cell and it is wrong on
every trial but one as soon as a search starts changing `fast`. As a method it
reads `self.fast_length`, so a legend cannot disagree with the array beside it.

Nothing requires it. A list built at the call site works exactly as well,
and the two charts below use both: the method for what the rule traded on, the
call site for what is being asked about it today.

In [3]:
class EmaCross(emsl.Strategy):

    def __init__(self, fast, slow, weight=0.95):    #  the tunables
        self.fast_length = fast
        self.slow_length = slow
        self.weight      = weight

    def init(self, engine):    #  emsl's hook, once per run
        #  engine.closes is a zero-copy view of the close column, so nothing is
        #  copied and nothing has to remember which column that is.
        self.fast = emsl.ta.ema(values=engine.closes, length=self.fast_length)
        self.slow = emsl.ta.ema(values=engine.closes, length=self.slow_length)

        self.up     = emsl.ta.crossover(values=self.fast, other=self.slow)
        self.down   = emsl.ta.crossunder(values=self.fast, other=self.slow)
        self.spread = self.fast - self.slow

        self.warmup = self.slow_length

    def next(self, state, engine):
        i = state["tick_index"]

        if state["position"] == 0.0 and self.up[i]:
            engine.market_buy(size=engine.qty_from_weight(fraction=self.weight))

        elif state["position"] > 0.0 and self.down[i]:
            engine.close()

    def marks(self):
        return [
            emsl.plot.Line(values=self.fast, name=f"EMA {self.fast_length}"),
            emsl.plot.Line(values=self.slow, name=f"EMA {self.slow_length}",
                           style="dashed"),
        ]

#### **Running it**

`Market` holds the venue in one object so a cost cannot be set in one place and
forgotten in another. **Its fee defaults are already realistic**, six basis
points taker and two maker. What defaults to zero is `slippage_bps` and
`impact`, and one order is allowed to eat a whole bar's volume. That is the
frictionless part, and it is the part that flatters a rule which trades often.

Setting the costs explicitly, even to values they already hold, turns an
assumption into a statement.

A printed Market shows only what differs from the defaults, which is why
the two fee lines are absent below even though they were passed in.

**An order decided on a bar fills on the next one.** That is the guard against
the most common backtesting mistake, deciding on a close and filling at that
same close. It also means `next` is never called on the final bar, and a
position still open at the end is marked into the return but closes no trade, so
it appears in no trade statistic.

`win_rate` is left out of the printout. It is meaningless on its own, and the
next section prints it beside the payoff, which is the only way it means
anything.

In [4]:
VENUE = emsl.Market(
    kind="spot",
    quote=10_000.0,
    fee_taker=0.0006,
    fee_maker=0.0002,
    slippage_bps=2.0,
)

strategy = EmaCross(fast=20, slow=60)
result   = VENUE.backtest(candles=candles).run(strategy=strategy)

print(VENUE)
print()

for key in ("total_return_pct", "sharpe", "max_drawdown_pct",
            "num_trades", "exposure_pct"):
    print(f"{key:<20} {result.stats[key]:>12,.2f}")

Market(kind='spot', slippage_bps=2.0)

total_return_pct           -39.12
sharpe                      -1.64
max_drawdown_pct            44.90
num_trades                  81.00
exposure_pct                48.83


#### **Where the money went**

**Read this before any ratio.** `decompose` splits the change in equity:
gross price PnL, fees, funding, and whatever is still open. It reports the
two costs as positive amounts paid away, so gross minus them plus the open
position is the whole of it. `summary` prints the same figures already
signed, as terms that add up to net, which is why one fee reads negative
there and positive here.

The question it answers is which of two failures happened. A rule can be
wrong about direction, or it can be right and hand the difference to the venue.
Those look identical in a return figure and completely different here. Compare
`gross_pnl` against `fees`: if gross is already negative, the costs were not the
problem and no amount of cheaper execution saves it. `fee_share` says what
fraction of the move went to the venue, and `turnover` says how many times the
account was traded through.

`breakeven_bps` asks it from the other side: how expensive would a round trip
have to be before this stops making money. It re-runs the strategy at rising
costs, so it takes the strategy rather than the finished result, and it refuses
fee settings of its own because sweeping the fee is the whole point. **`None`
means it was already losing at zero cost.**

`buy_and_hold` is the baseline that matters. A rule that made money in a year
the asset doubled has shown nothing yet, and `excess_return_pct` is the part
that was actually the rule rather than the market.

These print and also return. `summary` hands back the dict it printed
from and `compare` hands back its rows, so a report can reuse the numbers
instead of reparsing them. Naming the return is also what keeps a notebook
from dumping the whole dict under the table it just formatted.

In [5]:
printed = emsl.metrics.summary(result=result, frame=candles)

  return              -39.12 %
  sharpe               -1.64
    95% interval  -3.58 to 0.30   
  max drawdown         44.90 %

  gross pnl        -3,201.31
  fees               -710.66
  funding              -0.00
  still open           -0.00
  net              -3,911.97
  fee share             22.2 % of gross

  trades                  81
  long / short      81 /    0
  win rate              17.3 %
  payoff                2.36
  expectancy          -48.30 per trade
  worst streak            16 losses
  under water           96.6 % of bars
  longest              8,308 bars

  vs holding          -32.93 %


In [6]:
for label, amount in emsl.metrics.decompose(result=result).items():
    print(f"{label:<14} {amount:>14,.4f}")

held = emsl.metrics.buy_and_hold(result=result, frame=candles)

print(f"\nbuy and hold   {held['hold_return_pct']:>14,.2f} %")
print(f"excess         {held['excess_return_pct']:>14,.2f} %")

dies_at = emsl.metrics.breakeven_bps(
    strategy=EmaCross(fast=20, slow=60),
    data=candles,
)

if dies_at is None:
    print("breakeven        already losing at zero cost")
else:
    print(f"breakeven      {dies_at:>14,.1f} bps round trip")

gross_pnl         -3,201.3130
fees                 710.6593
funding                0.0000
unrealized            -0.0000
net               -3,911.9723
net_pct              -39.1197
fee_share              0.2220
turnover             118.4432

buy and hold            -6.19 %
excess                 -32.93 %
breakeven        already losing at zero cost


#### **Looking at it**

One call, and most of what appears was never asked for. **Passing `run=` brings
four things**: the fill arrows on the bars they filled on, the trade table under
the chart, an equity panel and a drawdown panel. The candles and the volume
panel come from the frame.

The two lines are the strategy's own arrays. `marks()` hands back
`self.fast` and `self.slow` by reference, not recomputed copies. A recomputed
average is a picture of a rule that was never executed, and it will agree with
the real one right up until the day it does not. Reaching for the attribute is
what makes that mistake impossible rather than merely discouraged.

They land on the price panel without being told to. **A bare array overlays the
candles unless doing so would cost them more than half the panel**, and gets its
own panel otherwise. That is a guess about the picture and never about your
intent: it may put a series somewhere you would not have, but it cannot quietly
flatten the candles to get it there. `panel=` overrides it either way, including
`panel="price"`.

**The chart will not render on GitHub**, which strips iframes out of notebook
output. It does survive the notebook being saved, closed and reopened with no
kernel and no network, because the whole document is the cell's output.

In [7]:
emsl.chart(
    frame=candles,
    marks=strategy.marks(),
    run=result,
    title=f"{EXCHANGE} {MARKET} {SYMBOL} {INTERVAL}, everything a run brings",
).show()

Everything here can be changed without re-running anything.

Each panel draws **A / L / %** in its own axis: auto, logarithmic, percent. Log
on the equity panel turns compounding into a straight line, which is the honest
way to look at a long run. Those are the same three values `scale=` takes, so
anything you find by clicking is something you can then pin in the call.

Percent has one trap. **It is per series, not per panel**, and each series is
baselined at its own first *visible* point. Two series that begin on different
bars are measured from different anchors while the viewport sits left of the
later one, and both the baseline and the numbers move as you pan. On a panel
with one curve that is exactly what you want. On a panel carrying a curve and a
benchmark that starts later, the two percentages are not comparable until you
scroll past the later start.

The trade table selects both ways. Click a bar inside a trade and its row
reveals; click a row and the chart frames that trade. The crosshair reads every
panel at once.

**The drawdown panel is not the equity panel as a percentage**, which is the
usual reason people reach to turn it off. Percent rescales an axis and leaves
the curve identical, so it says nothing new. Drawdown measures the fall from the
running peak, so it pins to zero on every new high and only moves while you are
below one. A run can be up on the year and well below its high on the same bar,
and the equity curve cannot show you the second number.

`strategy.marks() + [...]` puts two different owners in one list. The method
contributes the two averages the rule traded on. The spread, its zero line
and the signal arrows are questions about the run rather than parts of it,
and none of them belongs on the class. A list is a list, so the chart cannot
tell which came from where.

`panels=` **configures panels, it never creates or removes them.** A name it
does not recognise configures nothing, so a typo costs nothing rather than
manufacturing a blank pane. Listing panels also fixes their order against each
other, without moving them relative to panels you did not list.

A panel is created by being mentioned. `panel="signal"` on the spread line
is what brings that panel into existence; the `Panel(name="signal", ...)` below
only sets how tall it is. `weight` is a stretch factor rather than pixels, so a
layout holds at any size and the numbers only matter against each other.

`show=False` is the off switch, and it removes the panel and everything on
it. A hidden panel ships no data at all rather than going unpainted, so turning
off what you are not reading also makes the file smaller. Volume and drawdown
are off below because the viewport is on one trade, where neither answers
anything.

The spread is not price-scaled, so it gets its own panel and a `Level` at zero.
**Watch the two sets of arrows on the price panel: they are one bar apart.**
`Markers` draws the signal, on the bar the crossing happened. The engine's own
arrows draw the fill, on the bar after it. That gap is the fill rule made
visible.

**`focus=` moves the viewport and never slices the data**, so every marker stays
on its own bar. It takes a trade dict, a `(start, end)` pair of bar indices, or
an integer meaning the last N bars, and it clamps to the frame rather than
refusing.

In [8]:
worst = min(result.trades, key=lambda trade: trade["net_pnl"])

emsl.chart(
    frame=candles,
    marks=strategy.marks() + [
        emsl.plot.Line(values=strategy.spread, name="fast minus slow",
                       panel="signal"),
        emsl.plot.Level(value=0.0, panel="signal", style="dotted"),
        emsl.plot.Markers(mask=strategy.up, shape="arrow_up", offset=-18),
        emsl.plot.Markers(mask=strategy.down, shape="arrow_down", offset=18),
    ],
    run=result,
    panels=[
        emsl.plot.Panel(name="signal", weight=1.2),
        emsl.plot.Panel(name="equity", weight=2.0, scale="log"),
        emsl.plot.Panel(name="volume", show=False),
        emsl.plot.Panel(name="drawdown", show=False),
    ],
    focus=worst,
    title=f"the worst trade, {worst['net_pnl']:,.0f} quote",
).show()

#### **Closing**

Two lines in the output above are worth more than the return.

**`gross_pnl` against `fees`**, because it says which failure this was. If gross
is negative the rule was wrong about direction and cheaper execution would not
have saved it. If gross is positive and net is not, the idea was fine and the
trading paid for it. Only the second one is worth optimising.

**`excess_return_pct`**, because beating a falling market by losing less is
not the same as making money, and losing to a rising one while showing a
profit is not a strategy either. The number to look at is what the rule added
over holding.

The win rate is the number to distrust. It is the same figure for a rule that
grinds out small wins and for one that is quietly waiting for the bar that ends
it, which is why `summary` prints it beside the payoff and never on its own.

What is not here. This is the smallest complete loop, not the surface. The
engine also rests limit and stop orders, carries `reduce_only` and `post_only`
and a time in force, and lets a rule go short on a perpetual and pay funding
for the privilege. `emsl.ta` holds thirty indicators rather than the one used
above, and `emsl.metrics` holds a great deal more than the four calls here.

`client.close()` shuts down the background thread the client runs its async loop
on, and its connection pool.

In [9]:
client.close()